# Detector Variation Samples

- Note: Use EAF to run the event selection comparison part

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

# local imports
import sys
# sys.path.append('../../')
sys.path.append('/exp/sbnd/app/users/munjung/xsec/cafpyana_2026Jan17/cafpyana') # absolute path for running on EAF
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.categories import *
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.makedf.selections import *
from pyanalib.split_df_helpers import *
from pyanalib.pandas_helpers import *

import matplotlib.pyplot as plt 
import matplotlib as mpl
plt.style.use("presentation.mplstyle")

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [3]:
save_result = False
save_fig = save_result

save_fig_base_dir = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi"
today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "systematics-detvar-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

In [4]:
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10"

# variations
syst_keys = ["CV", "WireMod_XThetaXW", "WireMod_YZ"]
labels    = ["CV", r"X-$\theta_{xw}$", r"Y-Z"]
colors    = ["black", "C0", "C1"]

In [ ]:
syst_dfs = {}
for sidx, syst_key in enumerate(syst_keys):
    filename = "SystVar_{}_wtrks.df".format(syst_key)
    mc_file = path.join(file_dir, filename)
    mc_split_df = pd.read_hdf(mc_file, key="split")
    mc_n_split = get_n_split(mc_file)
    print("mc_n_split: %d" %(mc_n_split))
    print_keys(mc_file)

    n_max_concat = 100
    mc_keys2load = ['hdr', 'evt', 'trk'] 
    mc_dfs = load_dfs(mc_file, mc_keys2load, n_max_concat=n_max_concat)
    mc_hdr_df = mc_dfs['hdr']
    mc_evt_df = mc_dfs['evt']
    mc_trk_df = mc_dfs['trk']
    mc_trk_df = mc_trk_df[mc_trk_df.pfp.trk.producer != 4294967295]
    mask = (mc_trk_df.pfp.trk.len > 0) &\
         (mc_trk_df.pfp.pfochar.vtxdist < 100) #&\
    mc_trk_df = mc_trk_df[mask]

    nlevels = len(mc_evt_df.columns.levels)
    index_names = mc_evt_df.index.names
    mc_hdr_df.columns = pd.MultiIndex.from_tuples([tuple([str(c)] +[""] * (nlevels-1)) for c in mc_hdr_df.columns]) 
    mc_evt_df = multicol_merge(mc_evt_df.reset_index(), 
                               mc_hdr_df.reset_index(),
                               left_on=["__ntuple", "entry"],
                               right_on=["__ntuple", "entry"],
                               how="left"
                               ) 
    mc_evt_df = mc_evt_df.set_index(index_names, verify_integrity=True) 

    # need to match the nu_Es across files
    mc_evt_df["nu_E"] = mc_evt_df.mc.E

    syst_dfs[syst_key] = mc_evt_df
    syst_dfs[syst_key+"_trk"] = mc_trk_df

del mc_hdr_df
del mc_evt_df

In [ ]:
# # match common events
# for k in syst_keys:
#     syst_dfs[k] = syst_dfs[k].reset_index().set_index(["run","subrun","evt","nu_E"])
#     print(len(syst_dfs[k].index))

# for kidx, k in enumerate(syst_keys):
#     idxs = syst_dfs[k].index
#     if kidx == 0:
#         common_idxs = idxs
#     else:
#         common_idxs = common_idxs.intersection(idxs)
# print(len(common_idxs))

# for k in syst_keys:
#     syst_dfs[k] = syst_dfs[k].loc[common_idxs]

In [ ]:
# total POT (not used for syst sample comparison)
tot_pot = syst_dfs['CV'].groupby(level=[0,1]).nth(0)['pot'].sum()
print("mc_tot_pot: %.3e" %(tot_pot))

pot_str = "{:.2e}".format(tot_pot).replace("e+0", "e").replace("e+", "e")\
    .replace("e", " $\\times 10^{") + "}$"
pot_str = pot_str.replace(".00", "")
pot_str = pot_str.replace("$\\times 10^{", "$\\times 10^{")  
pot_str = pot_str.replace("}$", "}") 
pot_str = pot_str if pot_str.endswith("$") else pot_str + "$"
pot_str = pot_str.replace(" $", "$")
print(pot_str)

In [ ]:
# ==== events selection cuts ====
# slice cuts
nu_score_th   = 0.45
save_ntrks    = 2
# track quality cuts
trackscore_th = 0.5
vtxdist_th    = 1.2
# pid cuts
mu_chi2mu_th  = 30
mu_chi2p_th   = 100
mu_len_th     = 50
qual_th       = 0.2
p_chi2p_th    = 90
p_len_th      = 0
# kinematic cuts
mu_Plo_th     = 0.22
mu_Phi_th     = 1
p_Plo_th      = 0.3
p_Phi_th      = 1

In [ ]:
# sanity check to see if nus are matched correctly
evtdfs = [syst_dfs[syst_key] for syst_key in syst_keys]
var_name = ("mc","E")
bins = np.linspace(0.13, 4, 100)
plot_labels = ["Neutrino Energy", "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format("E")
n = variation_hists(evtdfs, 
                    var_name, 
                    bins,
                    colors, 
                    labels,
                    plot_labels,
                    vline=[],
                    save_fig=save_fig, save_name=save_name)


In [ ]:
for syst_key in syst_keys:
    syst_dfs[syst_key] = cut_clear_cosmic(syst_dfs[syst_key])
    syst_dfs[syst_key] = cut_vertex_in_fv(syst_dfs[syst_key])

In [ ]:
evtdfs = [syst_dfs[syst_key] for syst_key in syst_keys]
var_name = ("slc","nu_score")
bins = np.linspace(0., 1., 100)
plot_labels = ["nu_score", "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format("nu_score")
n = variation_hists(evtdfs, 
                    var_name, 
                    bins,
                    colors, 
                    labels,
                    plot_labels,
                    vline=[],
                    save_fig=save_fig, save_name=save_name)

In [ ]:
for syst_key in syst_keys:
    syst_dfs[syst_key] = cut_nu_score(syst_dfs[syst_key], nu_score_th)

In [ ]:
for syst_key in syst_keys:
    syst_dfs[syst_key] = syst_dfs[syst_key].reset_index().set_index(list(syst_dfs[f'{syst_key}_trk'].index.names)[:-1])
    syst_dfs[syst_key+"_trk"] = get_valid_trks(syst_dfs[syst_key+"_trk"])
    syst_dfs[syst_key+"_trk"] = match_trkdf_to_slcdf(syst_dfs[syst_key+"_trk"], syst_dfs[syst_key])

In [ ]:
evtdfs = [syst_dfs[syst_key+"_trk"] for syst_key in syst_keys]
var_name = ("pfp","pfochar","vtxdist")
bins = np.linspace(0, 10, 100)
plot_labels = ["vtx_dist", "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format("vtx_dist")
n = variation_hists(evtdfs, 
                    var_name, 
                    bins,
                    colors, 
                    labels,
                    plot_labels,
                    vline=[],
                    save_fig=save_fig, save_name=save_name)

In [ ]:
for k in syst_keys:
    syst_dfs[k] = get_trk_info(syst_dfs[k], syst_dfs[f'{k}_trk'], save_ntrks)
    del syst_dfs[f'{k}_trk']
del mc_trk_df

In [ ]:
evtdfs = [syst_dfs[syst_key] for syst_key in syst_keys]

var_name = "n_trks"
bins = np.linspace(1, 8, 8)
plot_labels = ["n_trks", "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format("n_trks")
n = variation_hists(evtdfs, 
                    var_name, 
                    bins,
                    colors, 
                    labels,
                    plot_labels,
                    vline=[],
                    save_fig=save_fig, save_name=save_name)

var_name = "n_good_trks"
bins = np.linspace(1, 8, 8)
plot_labels = ["n_good_trks", "Events (POT={})".format(pot_str), ""]
save_name = save_fig_dir + "/{}.png".format("n_good_trks")
n = variation_hists(evtdfs, 
                    var_name, 
                    bins,
                    colors, 
                    labels,
                    plot_labels,
                    vline=[],
                    save_fig=save_fig, save_name=save_name)

In [ ]:
for syst_key in syst_keys:
    syst_dfs[syst_key] = cut_2prong_contained(syst_dfs[syst_key])

In [ ]:
evtdf = [syst_dfs[syst_key] for syst_key in syst_keys]

for trk_id in range(1, 3):
    var_name = ("trk{}".format(trk_id), "pfp", "trackScore")
    bins = np.linspace(0, 1, 51)
    save_tag = "trk{}_trackScore".format(trk_id)
    plot_labels = [save_tag, "Events (POT={})".format(pot_str), ""]
    save_name = save_fig_dir + "/{}.png".format(save_tag)
    n = variation_hists(evtdfs, 
                        var_name, 
                        bins,
                        colors, 
                        labels,
                        plot_labels,
                        vline=[],
                        save_fig=save_fig, save_name=save_name)

In [ ]:
for syst_key in syst_keys:
    syst_dfs[syst_key] = cut_2prong_trackscore(syst_dfs[syst_key], trackscore_th)
    syst_dfs[syst_key] = cut_2prong_vtxdist(syst_dfs[syst_key], vtxdist_th)

In [ ]:
evtdf = [syst_dfs[syst_key] for syst_key in syst_keys]

for trk_id in range(1, 3):
    var_name = ("trk{}".format(trk_id), "pfp", "trk", "chi2pid", "I2", "chi2_muon")
    bins = np.linspace(0, 60, 61)
    save_tag = "trk{}_chi2_muon".format(trk_id)
    plot_labels = [save_tag, "Events (POT={})".format(pot_str), ""]
    save_name = save_fig_dir + "/{}.png".format(save_tag)
    n = variation_hists(evtdfs, 
                        var_name, 
                        bins,
                        colors, 
                        labels,
                        plot_labels,
                        vline=[],
                        save_fig=save_fig, save_name=save_name)

In [ ]:
evtdf = [syst_dfs[syst_key] for syst_key in syst_keys]

for trk_id in range(1, 3):
    var_name = ("trk{}".format(trk_id), "pfp", "trk", "chi2pid", "I2", "chi2_proton")
    bins = np.linspace(0, 300, 51)
    save_tag = "trk{}_chi2_proton".format(trk_id)
    plot_labels = [save_tag, "Events (POT={})".format(pot_str), ""]
    save_name = save_fig_dir + "/{}.png".format(save_tag)
    n = variation_hists(evtdfs, 
                        var_name, 
                        bins,
                        colors, 
                        labels,
                        plot_labels,
                        vline=[],
                        save_fig=save_fig, save_name=save_name)

In [ ]:
evtdf = [syst_dfs[syst_key] for syst_key in syst_keys]

for trk_id in range(1, 3):
    var_name = ("trk{}".format(trk_id), "pfp", "trk", "len")
    bins = np.linspace(0, 400, 51)
    save_tag = "trk{}_len".format(trk_id)
    plot_labels = [save_tag, "Events (POT={})".format(pot_str), ""]
    save_name = save_fig_dir + "/{}.png".format(save_tag)
    n = variation_hists(evtdfs, 
                        var_name, 
                        bins,
                        colors, 
                        labels,
                        plot_labels,
                        vline=[],
                        save_fig=save_fig, save_name=save_name)

In [ ]:
for syst_key in syst_keys:
    syst_dfs[syst_key] = get_mu_p_candidate(syst_dfs[syst_key], 
                            mu_chi2mu_th=mu_chi2mu_th, mu_chi2p_th=mu_chi2p_th, mu_len_th=mu_len_th, qual_th=qual_th,
                            p_chi2mu_th=-1, p_chi2p_th=p_chi2p_th, p_len_th=p_len_th)

    syst_dfs[syst_key] = cut_has_mu(syst_dfs[syst_key]) 
    syst_dfs[syst_key] = cut_mu_kinematics(syst_dfs[syst_key], mu_Plo_th=mu_Plo_th, mu_Phi_th=mu_Phi_th)

    syst_dfs[syst_key] = cut_has_p(syst_dfs[syst_key]) 
    syst_dfs[syst_key] = cut_p_kinematics(syst_dfs[syst_key], p_Plo_th=p_Plo_th, p_Phi_th=p_Phi_th)

In [ ]:
detvar_weights = {}

In [ ]:
syst_dfs["CV"][var_config.var_evt_reco_col]

In [ ]:
for var_config in [VariableConfig.muon_momentum(), VariableConfig.muon_direction(), VariableConfig.proton_momentum(), VariableConfig.proton_direction()]:
        evtdfs = [syst_dfs[syst_key] for syst_key in syst_dfs.keys()]
        var_name = var_config.var_evt_reco_col
        bins = var_config.bins
        plot_labels = [var_config.var_labels[1], "Events (POT={})".format(pot_str), ""]
        save_name = save_fig_dir + "/{}.png".format(var_config.var_save_name)
        n = variation_hists(evtdfs, 
                            var_name, 
                            bins,
                            colors, 
                            labels,
                            plot_labels,
                            vline=[],
                            save_fig=save_fig, save_name=save_name)
        detvar_weights[var_config.var_save_name] = [n[i]/n[0] for i in range(len(syst_dfs.keys()))]

In [ ]:
from pyanalib.variable_calculator import get_cc1p0pi_tki

tki_var_names = ["del_alpha", "del_phi", "del_Tp", "del_p", "del_Tp_x", "del_Tp_y"]
for syst_key in syst_dfs.keys():
    slc_mudf = syst_dfs[syst_key].mu.pfp.trk
    slc_pdf = syst_dfs[syst_key].p.pfp.trk
    slc_P_mu_col = pad_column_name(("P", "p_muon"), slc_mudf)
    slc_P_p_col = pad_column_name(("P", "p_proton"), slc_pdf)
    tki_reco = get_cc1p0pi_tki(slc_mudf, slc_pdf, slc_P_mu_col, slc_P_p_col)
    for var_name in tki_var_names:
        syst_dfs[syst_key] = multicol_add(syst_dfs[syst_key], tki_reco[var_name].rename(var_name))

In [ ]:
for var_config in [VariableConfig.tki_del_Tp(), VariableConfig.tki_del_Tp_x(), VariableConfig.tki_del_Tp_y(), VariableConfig.tki_del_alpha(), VariableConfig.tki_del_phi()]:
        evtdfs = [syst_dfs[syst_key] for syst_key in syst_dfs.keys()]
        var_name = var_config.var_evt_reco_col
        bins = var_config.bins
        plot_labels = [var_config.var_labels[1], "Events (POT={})".format(pot_str), ""]
        save_name = save_fig_dir + "/{}.png".format(var_config.var_save_name)
        n = variation_hists(evtdfs, 
                            var_name, 
                            bins,
                            colors, 
                            labels,
                            plot_labels,
                            vline=[],
                            save_fig=save_fig, save_name=save_name)
        detvar_weights[var_config.var_save_name] = [n[i]/n[0] for i in range(len(syst_dfs.keys()))]

# Detector Variation Systematic Uncertainty

In [ ]:
syst_name = "detvar"

In [ ]:
var_config = VariableConfig.muon_momentum()

In [ ]:
evtdfs = [syst_dfs[syst_key] for syst_key in syst_dfs.keys()]
var_name = var_config.var_evt_reco_col
bins = var_config.bins
plot_labels = [var_config.var_labels[1], "Events (POT={})".format(pot_str), ""]
approval = "internal"
save_name = save_fig_dir + "/{}.png".format(var_config.var_save_name)
n = variation_hists(evtdfs, 
                    var_name, 
                    bins,
                    colors, 
                    labels,
                    plot_labels,
                    approval=approval,
                    vline=[],
                    save_fig=save_fig, save_name=save_name)
detvar_weights[var_config.var_save_name] = [n[i]/n[0] for i in range(len(syst_dfs.keys()))]

In [ ]:
ret_dict = {}
for kidx, syst_key in enumerate(syst_dfs.keys()):
    if syst_key == "CV":
        continue

    # take syst variation as a unisim uncertainty
    cv_events = n[0]
    univ_events = np.array([n[kidx]]) 
    ret = get_covariance_matrix(univ_events, cv_events)
    ret_dict[syst_key] = ret
    plot_univ_hists(univ_events, 
                    cv_events,
                    syst_name, 
                    var_config)

    matrix_type = "cov"
    save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
    title = "{} {}".format(syst_key, matrix_type)
    plot_heatmap(ret[matrix_type], 
                 var_config, 
                 title=title, 
                 save_fig=save_fig, 
                 save_fig_name=save_fig_name)

    frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
    plot_frac_unc(frac_unc, var_config)

In [ ]:
# get total detector variation covariance matrix
for kidx, syst_key in enumerate(ret_dict.keys()):
    if kidx == 0:
        detvar_total_cov = ret_dict[syst_key]["cov_frac"]
    else:
        detvar_total_cov += ret_dict[syst_key]["cov_frac"]

frac_unc = np.sqrt(np.diag(detvar_total_cov))
plot_frac_unc(frac_unc, var_config)

In [ ]:
if 'syst_dict' not in locals():
    syst_dict = {}

syst_dict[var_config.var_save_name] = detvar_total_cov
for syst_key in ret_dict.keys():
    syst_dict[var_config.var_save_name + "_" + syst_key] = ret_dict[syst_key]["cov_frac"]

# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/detvar_syst_dict.npz", **syst_dict)

# sanity check: load saved file and check it agrees with the local one for all keys
loaded = np.load(file_dir + "/detvar_syst_dict.npz")
for key in syst_dict.keys():
    arr_local = syst_dict[key]
    arr_loaded = loaded[key]
    assert np.allclose(arr_local, arr_loaded), f"Mismatch for key: {key}"
print("All keys in syst_dict match the saved npz file.")